In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/bryanfu01/Reimplementation-of-REVERB-FL.git
%cd Reimplementation-of-REVERB-FL
!wget http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
!mkdir speech_command_dataset
!tar -xf speech_commands_v0.02.tar.gz -C speech_command_dataset

Cloning into 'Reimplementation-of-REVERB-FL'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 14 (delta 0), reused 14 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), 97.70 KiB | 24.42 MiB/s, done.
/content/Reimplementation-of-REVERB-FL
--2026-03-04 03:13:57--  http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz
Resolving download.tensorflow.org (download.tensorflow.org)... 64.233.170.207, 142.251.10.207, 142.251.12.207, ...
Connecting to download.tensorflow.org (download.tensorflow.org)|64.233.170.207|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2428923189 (2.3G) [application/gzip]
Saving to: ‘speech_commands_v0.02.tar.gz’

speech_commands_v0. 100%[===================>]   2.26G  23.3MB/s    in 1m 44s  

2026-03-04 03:15:41 (22.3 MB/s) - ‘speech_commands_v0.02.tar.gz’ saved [2428923189/2428923189]



In [ ]:
%cd /content/Reimplementation-of-REVERB-FL

!git pull

In [ ]:
!python main.py

In [ ]:
import torch
import matplotlib.pyplot as plt

def plot_defense_comparison(baseline_checkpoint_path, reverb_checkpoint_path, attack_name="PGD"):
    """
    Extracts accuracy histories from two saved checkpoints and plots them together.
    """
    # 1. Extract the histories directly from the saved files
    baseline_data = torch.load(baseline_checkpoint_path, map_location='cpu')
    reverb_data = torch.load(reverb_checkpoint_path, map_location='cpu')

    baseline_acc = baseline_data['acc_history']
    reverb_acc = reverb_data['acc_history']

    # 2. Convert to percentages and set up the X-axis
    baseline_pct = [acc * 100 for acc in baseline_acc]
    reverb_pct = [acc * 100 for acc in reverb_acc]

    # Assuming both ran for the same number of rounds
    rounds = range(1, len(baseline_acc) + 1)

    # 3. Create the figure
    plt.figure(figsize=(10, 6))

    # 4. Plot both lines with distinct styles
    # Red dashed line for the vulnerable baseline
    plt.plot(rounds, baseline_pct, marker='x', linestyle='--', color='r', label='Baseline FedAvg (No Defense)')

    # Solid blue line with dots for the robust defense
    plt.plot(rounds, reverb_pct, marker='o', linestyle='-', color='b', label='REVERB-FL (Active Defense)')

    # 5. Format to match academic paper standards
    plt.title(f'REVERB-FL vs Baseline FedAvg under {attack_name.upper()} Attack')
    plt.xlabel('Communication Round')
    plt.ylabel('Test Accuracy (%)')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.legend()

    # 6. Save and display
    filename = f'comparison_{attack_name}_vs_baseline.png'
    plt.savefig(filename, bbox_inches='tight')
    print(f"Comparison graph successfully saved as {filename}")
    plt.show()